# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 03 — Feature Engineering

**What this notebook does:**
1. Builds **driver features** — one row per driver per race (9 clustering-safe + 2 modeling-only)
2. Builds **circuit features** — one row per circuit (aggregated over training seasons 2019–2024 only)
3. Computes target variable (`position_gain`) and teammate pace delta

**Input:**  `data/processed/laps_clean.csv`, `data/processed/results_clean.csv`  
**Output:** `data/processed/driver_features.csv`, `data/processed/circuit_features.csv`

| Feature | Clustering-safe? | Description |
|---------|:-:|-------------|
| `avg_lap_time_norm` | ✅ | Mean normalized lap time |
| `lap_time_std` | ✅ | Std of normalized lap times |
| `best_lap_time_norm` | ✅ | Minimum normalized lap time |
| `avg_sector1/2/3_norm` | ✅ | Mean sector normalized times |
| `tyre_degradation_slope` | ✅ | Lap time increase per lap per stint |
| `avg_stint_length` | ✅ | Average laps per tyre stint |
| `number_of_pit_stops` | ✅ | Total pit stops in race |
| `teammate_pace_delta` | ❌ | Modeling only — controls for car quality |
| `position_gain` | ❌ | Target variable — never used for clustering |

In [ ]:
%pip install scipy --quiet

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports ready.')

## Step 1 — Configure Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
PROC_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'figures')

LAPS_CLEAN_PATH    = os.path.join(PROC_DIR, 'laps_clean.csv')
RESULTS_CLEAN_PATH = os.path.join(PROC_DIR, 'results_clean.csv')
DRIVER_FEAT_PATH   = os.path.join(PROC_DIR, 'driver_features.csv')
CIRCUIT_FEAT_PATH  = os.path.join(PROC_DIR, 'circuit_features.csv')

for path in [LAPS_CLEAN_PATH, RESULTS_CLEAN_PATH]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌ MISSING'} {os.path.basename(path)}")
    if not exists:
        print('     → Run 02_preprocessing.ipynb first!')

## Step 2 — Load Clean Data

In [ ]:
laps    = pd.read_csv(LAPS_CLEAN_PATH)
results = pd.read_csv(RESULTS_CLEAN_PATH)

# Standardize circuit name spelling across seasons
NAME_MAP = {
    'Monte Carlo': 'Monaco',
    'Sao Paulo':   'São Paulo',
    'Mexico':      'Mexico City',
}
laps['CircuitName']    = laps['CircuitName'].replace(NAME_MAP)
results['CircuitName'] = results['CircuitName'].replace(NAME_MAP)

print(f'✅ laps_clean:    {laps.shape[0]:,} rows × {laps.shape[1]} cols')
print(f'✅ results_clean: {results.shape[0]:,} rows × {results.shape[1]} cols')
print(f'\nSeasons: {sorted(laps["Season"].unique().tolist())}')
print(f'Circuits: {laps["CircuitName"].nunique()}')
print(f'Drivers:  {laps["Driver"].nunique()}')

## Step 3 — Tyre Degradation Slope Helper

Fits a linear regression of `LapTimeNorm` vs `LapNumber` per stint.
A steeper slope = faster tyre degradation = harder on tyres.

In [ ]:
def compute_tyre_degradation_slope(driver_race_laps):
    """Mean linear regression slope across all stints (min 3 laps per stint)."""
    slopes = []
    if 'Stint' not in driver_race_laps.columns:
        return np.nan
    for _, stint_data in driver_race_laps.groupby('Stint'):
        stint_data = stint_data.dropna(subset=['LapNumber', 'LapTimeNorm'])
        if len(stint_data) < 3:
            continue
        try:
            slope, *_ = stats.linregress(stint_data['LapNumber'].values, stint_data['LapTimeNorm'].values)
            slopes.append(slope)
        except Exception:
            continue
    return np.mean(slopes) if slopes else np.nan

print('✅ Helper function defined.')

## Step 4 — Build Driver Features

> ⏱️ The tyre degradation loop takes ~2–3 minutes.

In [ ]:
print('🔄 Computing driver features...')
driver_feat_rows = []

grouped = laps.groupby(['Season', 'RoundNumber', 'CircuitName', 'Driver'])
total   = len(grouped)

for i, ((season, round_num, circuit, driver), grp) in enumerate(grouped):
    if i % 500 == 0:
        print(f'  Progress: {i:,} / {total:,} ({i/total*100:.0f}%)')

    row = {
        'season':     season,
        'round':      round_num,
        'circuit_id': circuit,
        'driver_id':  driver,
    }

    row['avg_lap_time_norm']  = grp['LapTimeNorm'].mean()
    row['lap_time_std']       = grp['LapTimeNorm'].std()
    row['best_lap_time_norm'] = grp['LapTimeNorm'].min()

    for col, feat in [('Sector1TimeNorm', 'avg_sector1_norm'),
                      ('Sector2TimeNorm', 'avg_sector2_norm'),
                      ('Sector3TimeNorm', 'avg_sector3_norm')]:
        row[feat] = grp[col].mean() if col in grp.columns else np.nan

    row['tyre_degradation_slope'] = compute_tyre_degradation_slope(grp)

    if 'Stint' in grp.columns:
        stint_lengths = grp.groupby('Stint').size()
        row['avg_stint_length']    = stint_lengths.mean()
        row['number_of_pit_stops'] = max(0, len(stint_lengths) - 1)
    else:
        row['avg_stint_length']    = np.nan
        row['number_of_pit_stops'] = np.nan

    driver_feat_rows.append(row)

driver_features = pd.DataFrame(driver_feat_rows)

print(f'\n✅ Driver features built: {driver_features.shape[0]:,} rows × {driver_features.shape[1]} cols')
print(f'   Unique drivers:  {driver_features["driver_id"].nunique()}')
print(f'   Unique circuits: {driver_features["circuit_id"].nunique()}')

## Step 5 — Add Teammate Pace Delta & Target Variable

In [ ]:
# Teammate pace delta
team_map = results[['Season', 'RoundNumber', 'Abbreviation', 'TeamName']].copy()
team_map.columns = ['season', 'round', 'driver_id', 'team']

driver_features = driver_features.merge(team_map, on=['season', 'round', 'driver_id'], how='left')

team_avg = (
    driver_features
    .groupby(['season', 'round', 'team'])['avg_lap_time_norm']
    .mean()
    .rename('team_avg_pace')
    .reset_index()
)

driver_features = driver_features.merge(team_avg, on=['season', 'round', 'team'], how='left')
driver_features['teammate_pace_delta'] = driver_features['avg_lap_time_norm'] - driver_features['team_avg_pace']
driver_features = driver_features.drop(columns=['team_avg_pace'])

print(f'✅ teammate_pace_delta added  (mean: {driver_features["teammate_pace_delta"].mean():.4f}, should be ~0)')

In [ ]:
# Target variable
pos_gain = results[['Season', 'RoundNumber', 'Abbreviation', 'PositionGain', 'GridPosition', 'Position']].copy()
pos_gain.columns = ['season', 'round', 'driver_id', 'position_gain', 'grid_position', 'finish_position']

driver_features = driver_features.merge(pos_gain, on=['season', 'round', 'driver_id'], how='left')

print(f'✅ position_gain added')
print(f'   Null count: {driver_features["position_gain"].isna().sum()}')
print(f'   Range: {driver_features["position_gain"].min():.0f} to {driver_features["position_gain"].max():.0f}')

## Step 6 — Build Circuit Features

Aggregated over **training seasons only (2019–2024)** — no 2025 data here.

In [ ]:
TRAIN_SEASONS = [2019, 2020, 2021, 2022, 2023, 2024]

laps_train    = laps[laps['Season'].isin(TRAIN_SEASONS)].copy()
results_train = results[results['Season'].isin(TRAIN_SEASONS)].copy()
driver_feat_train = driver_features[driver_features['season'].isin(TRAIN_SEASONS)].copy()

print(f'Training laps:    {len(laps_train):,}')
print(f'Training results: {len(results_train):,}')
print(f'Training circuits: {laps_train["CircuitName"].nunique()}')

In [ ]:
print('🔄 Computing circuit features...')
circuit_feat_rows = []

for circuit, grp in laps_train.groupby('CircuitName'):
    row = {'circuit_id': circuit}

    row['lap_time_variability'] = grp['LapTimeNorm'].std()

    s1 = grp['Sector1Time'].mean() if 'Sector1Time' in grp.columns else np.nan
    s2 = grp['Sector2Time'].mean() if 'Sector2Time' in grp.columns else np.nan
    s3 = grp['Sector3Time'].mean() if 'Sector3Time' in grp.columns else np.nan
    total_t = (s1 or 0) + (s2 or 0) + (s3 or 0)

    row['sector1_dominance'] = s1 / total_t if total_t > 0 else np.nan
    row['sector2_dominance'] = s2 / total_t if total_t > 0 else np.nan
    row['sector3_dominance'] = s3 / total_t if total_t > 0 else np.nan

    # Strategy features from driver_feat_train
    circuit_driver_rows = driver_feat_train[driver_feat_train['circuit_id'] == circuit]

    row['avg_pit_stops_per_race'] = circuit_driver_rows['number_of_pit_stops'].mean()
    row['avg_stint_length']       = circuit_driver_rows['avg_stint_length'].mean()
    row['tyre_degradation_avg']   = circuit_driver_rows['tyre_degradation_slope'].mean()

    # Overtaking index
    circ_results = results_train[results_train['CircuitName'] == circuit]
    if len(circ_results) > 0:
        row['overtaking_index'] = (circ_results['GridPosition'] - circ_results['Position']).abs().mean()
        dnf_count = circ_results['Status'].str.contains('Retired|Accident|Collision|Engine|Mechanical',
                                                         case=False, na=False).sum()
        row['dnf_rate'] = dnf_count / len(circ_results)
    else:
        row['overtaking_index'] = np.nan
        row['dnf_rate']         = np.nan

    circuit_feat_rows.append(row)

circuit_features = pd.DataFrame(circuit_feat_rows)
print(f'\n✅ Circuit features built: {circuit_features.shape[0]} circuits × {circuit_features.shape[1]} cols')

## Step 7 — Save Feature Files

In [ ]:
driver_features.to_csv(DRIVER_FEAT_PATH, index=False)
circuit_features.to_csv(CIRCUIT_FEAT_PATH, index=False)

print(f'💾 driver_features.csv  → {driver_features.shape[0]:,} rows × {driver_features.shape[1]} cols')
print(f'💾 circuit_features.csv → {circuit_features.shape[0]} rows × {circuit_features.shape[1]} cols')
print(f'\n📊 Null counts in driver features:')
print(driver_features[[
    'avg_lap_time_norm', 'lap_time_std', 'tyre_degradation_slope',
    'avg_stint_length', 'position_gain', 'grid_position'
]].isna().sum().to_string())